# What the framework promises, and what each promise costs

The framework stands between two contracts.
To client code it promises what the summary of [client-authors.ipynb](client-authors.ipynb) lists: records never change, a value lives while something keeps it, a reader never sees a value change, and so on.
To bindings it promises when and how it calls them, as listed in section 7 of [workflow-authors.ipynb](workflow-authors.ipynb).

This notebook does not explain how the backend works inside.
It shows the promises at work on one example, seen from both sides at once, and then asks for each promise what keeping it costs: in the user's process, where the backend exists, and on the service, where it is designed but not built.
How the backend keeps them in this process is described in the docstring of `ess.dispatch.backend`.

## 1. One accumulator, seen from both sides

The driver of `IOFQ_MULTI` and the function of `FORWARD`, from `toy_sans.py`, are wrapped so that every call into them is noted with its time.
The calls of the client are noted when they return, in the left column; the calls into the bindings in the right column.
`FORWARD` is slowed to 300 ms, so that waiting for it shows.

In [1]:
import tempfile
import threading
import time
from pathlib import Path

from ess.dispatch import Backend, Client, Template, dataset
from ess.dispatch.testing import FakeDatasets, FakeStore
from ess.spec import Driver, DriverBinding, function_binding
from toy_sans import (
    CAN_RUNS,
    FORWARD,
    IOFQ_MULTI,
    SAMPLE_RUNS,
    ForwardOutputs,
    ForwardParams,
    IofQOutputs,
    forward,
    multi,
    single,
)

folder = tempfile.TemporaryDirectory()


class Trace:
    def __init__(self) -> None:
        self._start = time.monotonic()
        self._lines: list[tuple[float, str, str]] = []
        self._lock = threading.Lock()

    def note(self, side: str, what: str) -> None:
        with self._lock:
            self._lines.append((time.monotonic() - self._start, side, what))

    def show(self) -> None:
        print(f'{"time":>8}  {"client":34}  binding')
        for t, side, what in self._lines:
            indent = '' if side == 'client' else ' ' * 36
            print(f'{1000 * t:5.0f} ms  {indent}{what}')


class TracedDriver:
    def __init__(self, driver: Driver[IofQOutputs], trace: Trace) -> None:
        self._driver, self._trace, self._n = driver, trace, 0

    def push(self, table: str, rows: list) -> None:
        self._n += 1
        self._trace.note('binding', f'push {self._n} into {table} starts')
        self._driver.push(table, rows)

    def outputs(self) -> IofQOutputs:
        self._trace.note('binding', f'outputs() of state {self._n}')
        return self._driver.outputs()


def traced_client(trace: Trace) -> Client:
    datasets = FakeDatasets(proposal='p1', directory=Path(folder.name))
    for run, data in {**SAMPLE_RUNS, **CAN_RUNS}.items():
        datasets.measure(run, data)

    def make_driver(fixed: dict) -> TracedDriver:
        trace.note('binding', 'make_driver()')
        return TracedDriver(multi.make_driver(fixed), trace)

    def slow_forward(params: ForwardParams) -> ForwardOutputs:
        trace.note('binding', 'FORWARD starts')
        time.sleep(0.3)
        trace.note('binding', 'FORWARD returns')
        return forward(params)

    bind = [
        single,
        DriverBinding(IOFQ_MULTI, make_driver),
        function_binding(FORWARD, slow_forward),
    ]
    backend = Backend(datasets, bind, store=FakeStore())
    return Client(backend, proposal='p1', submitter='alice', owns_backend=True)

The client code: four pushes, a request that reads state 2, a read of state 3, and a freeze of state 4.

In [2]:
trace = Trace()
client = traced_client(trace)


def note(what: str) -> None:
    trace.note('client', what)


total = client.accumulator(Template(IOFQ_MULTI, blanks=('sample', 'can')))
note('accumulator() returns')
total.push(can=[dataset(run=110)])
note('push 1 returns')
total.push(sample=[{'run': dataset(run=101)}])
note('push 2 returns')
first = client.submit(FORWARD, params={'curve': total.ref('iofq')})
note('submit(FORWARD) returns: pins state 2')
total.push(sample=[{'run': dataset(run=102)}])
note('push 3 returns')
state_3 = client.output(total, 'iofq')
note('output() returns: pinned state 3, copied')
total.push(sample=[{'run': dataset(run=103)}])
note('push 4 returns')
frozen = client.freeze(total)
note('freeze() returns: pins state 4')
client.wait([first, frozen])
note('wait() returns')
trace.show()
client.close()

    time  client                              binding
    4 ms  accumulator() returns
    4 ms                                      make_driver()
    8 ms  push 1 returns
    8 ms  push 2 returns
    9 ms  submit(FORWARD) returns: pins state 2
    9 ms  push 3 returns
   74 ms                                      push 1 into can starts
   78 ms                                      push 2 into sample starts
   82 ms                                      outputs() of state 2
   84 ms                                      FORWARD starts
  384 ms                                      FORWARD returns
  385 ms                                      push 3 into sample starts
  389 ms                                      outputs() of state 3
  393 ms  output() returns: pinned state 3, copied
  394 ms  push 4 returns
  395 ms  freeze() returns: pins state 4
  396 ms                                      push 4 into sample starts
  404 ms                                      outputs() of state 4
  407

Reading the trace from top to bottom:

- **Every call that starts work returns at once.** `accumulator`, `push`, `submit`, and `freeze` return once the backend has logged them. The binding works later, on the backend's workers.
- **Pinning never waits.** `submit(FORWARD)` pins state 2 and returns without waiting for push 2 to be added; FORWARD then waits for state 2.
- **Push n+1 waits for the readers of state n**: the requests that read its outputs, until their workflow has returned, and the reads by `client.output`, until they have copied. Push 3 starts only once FORWARD has returned, since FORWARD reads the outputs of state 2 without a copy, and the driver may add push 3 in place.
- **`client.output` waits for its state**, here state 3, and copies it, so the client holds a value that later pushes cannot change.
- **`outputs()` is called once per state that is read.** State 1, with no sample row, is never read.
- **`make_driver` is called once**, and the pushes come one at a time, in the order logged.

The client sees the same order as if every call ran after the one before it; the binding sees a driver that only moves forward.

One more promise, and its limit: cancelling ends a record at once, but a workflow that has started runs on.

In [3]:
trace = Trace()
client = traced_client(trace)
reference = client.compute(
    IOFQ_MULTI,
    params={'sample': [{'run': dataset(run=101)}], 'can': [dataset(run=110)]},
)
slow = client.submit(FORWARD, params={'curve': reference.ref('iofq')})
time.sleep(0.05)
client.cancel(slow)
note(f'cancel() returns: the record is {client.status(slow).value}')
time.sleep(0.4)
trace.show()

    time  client                              binding
    5 ms                                      make_driver()
   14 ms                                      push 1 into sample starts
   19 ms                                      push 2 into can starts
   25 ms                                      outputs() of state 2
   29 ms                                      FORWARD starts
   80 ms  cancel() returns: the record is cancelled
  329 ms                                      FORWARD returns


## 2. The promises, and what keeping them costs

The first two columns are the contract; the last two are the price.
"This process" is `local()`, a backend in the user's process, which exists.
"Service" is the backend that DMSC, the ESS data centre, hosts for batch and automatic reduction, which is designed only.
Letters and numbers such as D3 name stories in [system-stories.md](../system-stories.md).

| Promise | Relied on by | In this process | On the service |
|---|---|---|---|
| A record never changes; only its status does. | clients, provenance | records are entries in an append-only log (ADR 0004) | the same append-only lists, stored durably in a form the service chooses (ADR 0004); a new version must read an old one's records (H2) |
| A refused call submits nothing, and names the field. | apps (forms), batches | every request of a call is checked before any is logged, without reading data: each inline value by its field, each reference by its fit to its parameter's type | the same; the service needs every spec it accepts |
| A workflow gets each value as the type its parameter declares, whether given inline or by reference. | bindings | the backend reads each reference when the workflow runs, checks each value by its parameter's field alone, and builds the params instance without running its model validators | the data must be read where the workflow runs |
| A request runs once the records it references have completed. | chains | the only scheduling there is | a request must reach the data it references where it runs |
| The rows of a table are distinct, also by the rows read. | accumulators: a time slice is never summed twice | a push is compared with the table as the records hold it when it is logged, and, for rows with a JSON form, by the JSON forms of the rows read when it is added; a repeat found then fails the push and stops the accumulator | the JSON forms read so far are kept where the driver is |
| A value lives while something keeps it, and only then; work that nothing keeps is cancelled. | clients (memory), the service (bounded memory) | the backend counts keepers per value | a memory cap per client, and a lease that ends a client that makes no call (ADR 0002); how the cap is set and how a vanished client is noticed are open (scipp/essapps#27) |
| Persisted values outlive the client; history goes as a whole once the proposal is idle. | batch and automatic reduction | a store passed to `local`; history ends with the process | a store per proposal; when and by whom values are dropped from it before the history is, to free disk, is open (ADR 0002, H1, H3) |
| A reader never sees a value change: push n+1 waits for the readers of state n. | clients that read states; drivers that add in place | a wait on one driver in one process | a request that reads a state must run where the driver is, or get a copy (ADR 0003; README open question 5) |
| Pinning never waits. | controllers in loops | a pin is the number of pushes logged | the same |
| A call through a stage, and a state of an accumulator, make the record of their plain request. | provenance, rerunning | the `sciline.Stage` and the driver keep the symmetry; `local()` never restarts, and a backend started on a file log reruns a call through a stage as its plain request, and fails the record of a state | the calls through a stage must run where the binding's stage is held |
| Where a request runs changes neither its record nor its outputs. | everybody | one place | the same software everywhere; records name no software versions (F2) |
| A proposal reads only its own datasets and records. | DMSC (G5) | the backend checks every reference against the client's proposal | the same check behind authentication; a modified client cannot get around it |
| Cancelling ends a record at once. | batches restarted after a mistake (D3) | a running workflow is not interrupted | its worker stays busy until it returns, and the resubmitted batch waits (D3 gap) |

To bindings the framework promises the calls of the table in section 7 of [workflow-authors.ipynb](workflow-authors.ipynb).
Their cost: one driver takes one push at a time, on one worker.
Spreading one accumulator over several workers would need a merge of two drivers, which the driver protocol does not offer (README open question 1).

## 3. Which promises are proven only in this process

Every promise above has one implementation, the backend in the user's process.
The ones whose service column is open are where the contract may still have to change:

- **Where a driver lives.** "A reader never sees a value change" costs one wait in one process. On the service, readers of a state on other workers need a copy, or must run where the driver is.
- **Memory.** "A value lives while something keeps it" bounds memory in the user's process by what the user keeps. The service bounds it per client with a cap and a lease (ADR 0002); how the cap is set, and where a client's values are held when its work spreads over nodes, are open (scipp/essapps#27).
- **Copies.** In the user's process, `client.output` of a record the client keeps returns the value itself, not a copy. Notebooks with large results profit, and modifying the value in place is unguarded (README open question 7). On the service every read is a copy, which `select=` limits to a slice (B4).
- **Cancelling.** A cancelled workflow that has started keeps its worker.
- **Software versions.** Placement and rerunning a record assume the same software, which records do not name.

The rest, the log, the keepers, and the waits, are implementation details as long as these promises hold.

In [4]:
client.close()
folder.cleanup()